# **Load Pre-trained Toxicity Model**
In this step, a pre-trained toxicity classification model is loaded from Hugging Face and tests it on a non‑toxic review (“I love this game!”). The output is a label (NON_TOXIC) and a confidence score.




In [ ]:
from transformers import pipeline

# تحميل النموذج. قد يستغرق هذا الأمر بعض الوقت لأول مرة.
classifier = pipeline("text-classification", model="cooperleong00/deberta-v3-large_toxicity-scorer")

# مثال على الاستخدام لمراجعة واحدة
result = classifier("I love this game!")  # مراجعة غير سامة
print(result)  # من المتوقع: [{'label': 'NON_TOXIC', 'score': 0.99}]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cooperleong00/deberta-v3-large_toxicity-scorer
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'label': 'LABEL_0', 'score': 0.9731782674789429}]


# **Data Upload & Exploration**
The dataset is uploaded, extracted, and explored by inspecting its structure and columns to ensure it is ready for processing.

In [ ]:
# Opens a file upload widget in Google Colab to upload files manually
from google.colab import files
uploaded = files.upload()

Saving archive.zip to archive.zip


In [ ]:
#Unzips a file named archive.zip in the current Colab environment
!unzip archive.zip

Archive:  archive.zip
  inflating: output.csv              
  inflating: output_steamspy.csv     


In [ ]:
#Lists all files and directories in the current working directory
import os
os.listdir()

['.config', 'output_steamspy.csv', 'archive.zip', 'output.csv', 'sample_data']

In [ ]:
#Reads a CSV file named output.csv into a Pandas DataFrame and displays the first few rows
import pandas as pd

df = pd.read_csv("output.csv")
df.head()

,id,app_id,content,author_id,is_positive
0,181331361,100,At least its a counter strike -1/100,76561199556485100,Negative
1,180872601,100,Uh... So far my playthrough has not been great...,76561199230620391,Negative
2,177836246,100,Better mechanics than cs2,76561198417690647,Negative
3,177287444,100,buggy mess and NOT fun to play at all,76561199077268730,Negative
4,176678990,100,"Whoever came up with this, is gonna fucking ge...",76561199104544266,Negative


In [ ]:
# Prints the column names of the DataFrame to check the available fields
df.columns

Index(['id', 'app_id', 'content', 'author_id', 'is_positive'], dtype='object')

# **Initial Data Processing Pipeline**
Full pipeline for the first stage: loads output.csv, cleans the reviews, balances the dataset to 5000 positive and 5000 negative reviews, then uses a binary toxicity classifier (deberta-v3-large_toxicity-scorer) to label each review as TOXIC or NON_TOXIC. Finally, Saves the results to binary_toxic_results.csv for model training

In [ ]:
# ============================================================
# Full Code: Analyze 10,000 Steam Reviews from output.csv
# 1) Load data directly from Colab without manual upload
# 2) Use the actual columns: content, is_positive
# 3) Clean the text data
# 4) Balance the dataset to 5,000 Positive and 5,000 Negative samples
# 5) Perform binary classification: Toxic / Non-Toxic
# 6) Save only the binary classification results
# ============================================================

# 1) Install required libraries
!pip install -q pandas numpy scikit-learn transformers datasets torch accelerate

# 2) Import libraries
import pandas as pd
import numpy as np
import re
import os
import warnings
from sklearn.utils import resample
from sklearn.metrics import accuracy_score
from transformers import pipeline

warnings.filterwarnings("ignore")

print("=" * 70)
print("الخطوة 0: تحميل البيانات مباشرة من output.csv")
print("=" * 70)

# 3) Load the file directly without manual upload
file_name = "output.csv"

if not os.path.exists(file_name):
    raise FileNotFoundError(f" الملف {file_name} غير موجود في بيئة Colab")

df = pd.read_csv(file_name)
print(f" تم تحميل الملف: {file_name}")
print(f" عدد الصفوف الأصلي: {len(df)}")

print("\nأسماء الأعمدة الموجودة:")
print(df.columns.tolist())

# 4) Ensure the actual columns exist
required_cols = ['content', 'is_positive']
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"العمود المطلوب غير موجود: {col}")

# 5) Select only the important columns
df = df[['content', 'is_positive']].copy()
df.columns = ['raw_review', 'original_label']

print("\n تم اختيار الأعمدة المطلوبة")
print(df.head(3))

# 6) Normalize label values
df['original_label'] = df['original_label'].astype(str).str.strip().str.lower()

label_map = {
    'positive': 'Positive',
    'negative': 'Negative'
}

df = df[df['original_label'].isin(label_map.keys())].copy()
df['label'] = df['original_label'].map(label_map)

print("\n بعد توحيد الليبل:")
print(df['label'].value_counts())

# 7) Clean the text data
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)   # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)              # Keep only English alphabet characters
    text = re.sub(r'\s+', ' ', text).strip()              # Remove unnecessary extra spaces
    return text

df['clean_review'] = df['raw_review'].apply(clean_text)

# Remove empty or very short texts
df = df[df['clean_review'].str.len() > 3].copy()

print("\n بعد التنظيف:")
print(f"عدد المراجعات المتبقية: {len(df)}")

# 8) Split the data
positive = df[df['label'] == 'Positive'].copy()
negative = df[df['label'] == 'Negative'].copy()

print("\n" + "=" * 70)
print("الخطوة 1: موازنة البيانات إلى 10000 مراجعة")
print("=" * 70)
print(f"عدد Positive المتوفر: {len(positive)}")
print(f"عدد Negative المتوفر: {len(negative)}")

def balance_to_n(df_class, n, class_name):
    if len(df_class) >= n:
        return resample(df_class, replace=False, n_samples=n, random_state=42)
    else:
        print(f"عدد {class_name} أقل من {n}، سيتم التكرار للوصول إلى {n}")
        return resample(df_class, replace=True, n_samples=n, random_state=42)

# Important adjustment: 5000 + 5000 = 10,000
positive_balanced = balance_to_n(positive, 5000, "Positive")
negative_balanced = balance_to_n(negative, 5000, "Negative")

df_balanced = pd.concat([positive_balanced, negative_balanced], axis=0)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f" بعد الموازنة: {len(df_balanced)} عينة")
print(df_balanced['label'].value_counts())

# 9) Binary classification: Toxic / Non-Toxic
print("\n" + "=" * 70)
print("الخطوة 2: التصنيف الثنائي (Toxic / Non-Toxic)")
print("=" * 70)

print(" جاري تحميل نموذج التصنيف الثنائي...")
binary_classifier = pipeline(
    "text-classification",
    model="cooperleong00/deberta-v3-large_toxicity-scorer"
)
print(" تم تحميل النموذج الثنائي")

batch_size = 32
texts = df_balanced['clean_review'].tolist()
all_predictions = []

print(" جاري تصنيف 10000 مراجعة...")
for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    preds = binary_classifier(batch)
    all_predictions.extend(preds)

df_balanced['toxic_label'] = [pred['label'] for pred in all_predictions]
df_balanced['toxic_score'] = [pred['score'] for pred in all_predictions]

# Convert labels to 0 / 1
df_balanced['is_toxic'] = df_balanced['toxic_label'].apply(
    lambda x: 1 if ('LABEL_1' in str(x).upper() or 'TOXIC' in str(x).upper()) else 0
)

# 10) Calculate proxy accuracy (approximate only)
df_balanced['true_binary'] = df_balanced['label'].apply(
    lambda x: 'Non-Toxic' if x == 'Positive' else 'Toxic'
)
df_balanced['pred_binary'] = df_balanced['is_toxic'].apply(
    lambda x: 'Toxic' if x == 1 else 'Non-Toxic'
)

proxy_accuracy = accuracy_score(df_balanced['true_binary'], df_balanced['pred_binary'])

print(f"\n Proxy Accuracy (تقريبية فقط): {proxy_accuracy:.4f} ({proxy_accuracy*100:.2f}%)")
print(f"عدد المراجعات المصنفة Toxic: {df_balanced['is_toxic'].sum()}")
print(f"عدد المراجعات المصنفة Non-Toxic: {len(df_balanced) - df_balanced['is_toxic'].sum()}")

print("\nعينة من نتائج التصنيف الثنائي:")
print(df_balanced[['clean_review', 'label', 'toxic_label', 'toxic_score']].head(10))

# 11) Save only the binary results
binary_columns = [
    'raw_review',
    'clean_review',
    'label',
    'toxic_label',
    'toxic_score',
    'is_toxic'
]

df_binary = df_balanced[binary_columns].copy()
df_binary.to_csv("binary_toxic_results.csv", index=False)

print("\n" + "=" * 70)
print("تم حفظ النتائج الثنائية")
print("=" * 70)
print("تم حفظ الملف: binary_toxic_results.csv")

print("\nعينة من الملف النهائي للمرحلة الأولى:")
print(df_binary.head(10))

print("\n انتهت الخطوة الأولى بنجاح")
print(" الملف الناتج:")
print("- binary_toxic_results.csv")

الخطوة 0: تحميل البيانات مباشرة من output.csv
✅ تم تحميل الملف: output.csv
✅ عدد الصفوف الأصلي: 201151

أسماء الأعمدة الموجودة:
['id', 'app_id', 'content', 'author_id', 'is_positive']

✅ تم اختيار الأعمدة المطلوبة
                                          raw_review original_label
0               At least its a counter strike -1/100       Negative
1  Uh... So far my playthrough has not been great...       Negative
2                          Better mechanics than cs2       Negative

✅ بعد توحيد الليبل:
label
Positive    102660
Negative     98491
Name: count, dtype: int64

✅ بعد التنظيف:
عدد المراجعات المتبقية: 185502

الخطوة 1: موازنة البيانات إلى 10000 مراجعة
عدد Positive المتوفر: 94271
عدد Negative المتوفر: 91231
✅ بعد الموازنة: 10000 عينة
label
Negative    5000
Positive    5000
Name: count, dtype: int64

الخطوة 2: التصنيف الثنائي (Toxic / Non-Toxic)
⏳ جاري تحميل نموذج التصنيف الثنائي...


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cooperleong00/deberta-v3-large_toxicity-scorer
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ تم تحميل النموذج الثنائي
⏳ جاري تصنيف 10000 مراجعة...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



🎯 Proxy Accuracy (تقريبية فقط): 0.5945 (59.45%)
عدد المراجعات المصنفة Toxic: 2015
عدد المراجعات المصنفة Non-Toxic: 7985

عينة من نتائج التصنيف الثنائي:
                                        clean_review     label toxic_label  \
0  the only reason i won t recommend this excelle...  Negative     LABEL_0   
1                        no doubt the best csgo game  Positive     LABEL_0   
2                                               surf  Positive     LABEL_0   
3  best ww game like wow imagin fighting for nazi...  Positive     LABEL_0   
4                                               tack  Positive     LABEL_0   
5                 overrated piece of crap from valve  Negative     LABEL_1   
6                 portals lasers puzzles robots rice  Positive     LABEL_0   
7                          cannot play with one hand  Negative     LABEL_0   
8                                           igra kal  Negative     LABEL_0   
9  spice up this golden masterpiece with the enha...  Positive     

# **Multi-label Model Training (RoBERTa)**
trains a multi‑label RoBERTa model (labels: THREAT, BULLYING, SEXUAL_HARASSMENT, HATE_SPEECH) on an Excel file with 1000 labeled examples. Then loads the binary results from step 1 and applies the multi‑label model only on reviews that were classified as TOXIC. The final output is saved as steam_final_multilabel_roberta_10k.xlsx.

In [ ]:
# ============================================================
# FULL PIPELINE - FINAL VERSION
# 1) Train multi-label classifier on labeled Excel data
#    using roberta-base
# 2) Load binary_toxic_results.csv from step 1
# 3) Predict ONLY on rows where is_toxic = 1
#
# Final labels:
# THREAT, BULLYING, SEXUAL_HARASSMENT, HATE_SPEECH
#
# Each final label output is ONLY 0 or 1
# is_toxic remains from the first model only
# ============================================================

!pip install -q pandas openpyxl scikit-learn transformers datasets accelerate sentencepiece torch

import os
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# ============================================================
# 1) FILE PATHS
# ============================================================
excel_file = "/content/sample_data/gaming_toxicity_dataset_1000_v2.xlsx"
binary_file = "/content/binary_toxic_results.csv"   # Become CSV
sheet_name = "Data"

# ============================================================
# 2) CHECK FILES
# ============================================================
if not os.path.exists(excel_file):
    raise FileNotFoundError(f" Training Excel file not found: {excel_file}")

if not os.path.exists(binary_file):
    raise FileNotFoundError(f"Binary results file not found: {binary_file}")

# ============================================================
# 3) LOAD LABELED TRAINING DATA
# ============================================================
train_df = pd.read_excel(excel_file, sheet_name=sheet_name)

print("=" * 70)
print("STEP 1: Load labeled training data from Excel")
print("=" * 70)
print("Columns found:")
print(train_df.columns.tolist())

required_train_cols = [
    "comment",
    "threat",
    "bullying",
    "sexual_harassment",
    "hate_speech"
]

for col in required_train_cols:
    if col not in train_df.columns:
        raise ValueError(f" Missing required training column: {col}")

train_df = train_df[required_train_cols].copy()

# clean text
train_df["comment"] = train_df["comment"].astype(str).fillna("").str.strip()
train_df = train_df[train_df["comment"].str.len() > 0].copy()

# force labels to float
label_cols = ["threat", "bullying", "sexual_harassment", "hate_speech"]
for col in label_cols:
    train_df[col] = train_df[col].astype(float)

print(f"\n Training rows after cleaning: {len(train_df)}")
print("\nLabel distribution:")
for col in label_cols:
    print(f"{col}: {int(train_df[col].sum())}")

# ============================================================
# 4) SPLIT TRAIN / VALIDATION
# ============================================================
train_part, val_part = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("\nTrain size:", len(train_part))
print("Validation size:", len(val_part))

# ============================================================
# 5) MODEL + TOKENIZER
# ============================================================
model_name = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

id2label = {
    0: "THREAT",
    1: "BULLYING",
    2: "SEXUAL_HARASSMENT",
    3: "HATE_SPEECH"
}
label2id = {v: k for k, v in id2label.items()}

def tokenize_function(batch):
    return tokenizer(
        batch["comment"],
        truncation=True,
        max_length=256
    )

def make_hf_dataset(df):
    ds = Dataset.from_pandas(df.reset_index(drop=True))

    # tokenize
    ds = ds.map(tokenize_function, batched=True)

    # create labels correctly as float list
    def build_labels(example):
        example["labels"] = [
            float(example["threat"]),
            float(example["bullying"]),
            float(example["sexual_harassment"]),
            float(example["hate_speech"])
        ]
        return example

    ds = ds.map(build_labels)

    # keep only needed columns
    keep_cols = ["input_ids", "attention_mask", "labels"]
    ds = ds.remove_columns([c for c in ds.column_names if c not in keep_cols])

    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

train_ds = make_hf_dataset(train_part)
val_ds = make_hf_dataset(val_part)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ============================================================
# 6) METRICS
# ============================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    labels = np.array(labels)
    if labels.ndim == 1:
        labels = labels.reshape(-1, 4)

    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    subset_acc = accuracy_score(labels, preds)

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "subset_accuracy": subset_acc
    }

# ============================================================
# 7) TRAINING ARGUMENTS
# ============================================================
training_args = TrainingArguments(
    output_dir="./roberta_multilabel_model",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    report_to="none",
    fp16=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# ============================================================
# 8) TRAIN MODEL
# ============================================================
print("\n" + "=" * 70)
print("STEP 2: Training RoBERTa multi-label model on labeled Excel file")
print("=" * 70)

trainer.train()

print("\n" + "=" * 70)
print("STEP 3: Validation metrics")
print("=" * 70)

eval_results = trainer.evaluate()
print(eval_results)

# ============================================================
# 9) LOAD BINARY RESULTS FROM STEP 1
# ============================================================
df_balanced = pd.read_csv(binary_file)   # Become read_csv

required_binary_cols = [
    "raw_review",
    "clean_review",
    "label",
    "is_toxic",
    "toxic_score"
]

for col in required_binary_cols:
    if col not in df_balanced.columns:
        raise ValueError(f"Missing required binary column: {col}")

print("\n" + "=" * 70)
print("STEP 4: Load binary_toxic_results.csv")
print("=" * 70)
print(f"Total rows: {len(df_balanced)}")
print(f"Toxic rows: {int(df_balanced['is_toxic'].sum())}")

# ============================================================
# 10) PREPARE FINAL COLUMNS
# ============================================================
final_multi_cols = ["THREAT", "BULLYING", "SEXUAL_HARASSMENT", "HATE_SPEECH"]
for col in final_multi_cols:
    df_balanced[col] = 0

# ============================================================
# 11) PREDICT ONLY ON TOXIC REVIEWS
# ============================================================
toxic_reviews = df_balanced[df_balanced["is_toxic"] == 1].copy()

print("\n" + "=" * 70)
print("STEP 5: Predict only for rows where is_toxic = 1")
print("=" * 70)
print(f"Toxic rows to classify: {len(toxic_reviews)}")

if len(toxic_reviews) > 0:
    toxic_reviews = toxic_reviews.reset_index().rename(columns={"index": "orig_index"})

    toxic_dataset_df = toxic_reviews[["orig_index", "clean_review"]].copy()
    toxic_dataset_df = toxic_dataset_df.rename(columns={"clean_review": "comment"})
    toxic_dataset_df["comment"] = toxic_dataset_df["comment"].astype(str).fillna("").str.strip()

    toxic_ds = Dataset.from_pandas(toxic_dataset_df)

    toxic_ds = toxic_ds.map(tokenize_function, batched=True)

    keep_cols = ["orig_index", "input_ids", "attention_mask"]
    toxic_ds = toxic_ds.remove_columns([c for c in toxic_ds.column_names if c not in keep_cols])
    toxic_ds.set_format(type="torch", columns=["orig_index", "input_ids", "attention_mask"])

    raw_preds = trainer.predict(toxic_ds)
    logits = raw_preds.predictions

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    for i, row in toxic_reviews.iterrows():
        original_idx = int(row["orig_index"])

        df_balanced.at[original_idx, "THREAT"] = int(preds[i][0])
        df_balanced.at[original_idx, "BULLYING"] = int(preds[i][1])
        df_balanced.at[original_idx, "SEXUAL_HARASSMENT"] = int(preds[i][2])
        df_balanced.at[original_idx, "HATE_SPEECH"] = int(preds[i][3])

print(" Finished multi-label prediction")

# ============================================================
# 12) BUILD final_label
# ============================================================
def get_final_label(row):
    if int(row["is_toxic"]) == 0:
        return "NON_TOXIC"

    labels = []
    if int(row["THREAT"]) == 1:
        labels.append("THREAT")
    if int(row["BULLYING"]) == 1:
        labels.append("BULLYING")
    if int(row["SEXUAL_HARASSMENT"]) == 1:
        labels.append("SEXUAL_HARASSMENT")
    if int(row["HATE_SPEECH"]) == 1:
        labels.append("HATE_SPEECH")

    if len(labels) == 0:
        return "TOXIC_OTHER"

    return ", ".join(labels)



# ============================================================
# 13) SAVE FINAL OUTPUT (WITHOUT final_label)
# ============================================================

final_columns = [
    "raw_review",
    "clean_review",
    "label",
    "is_toxic",
    "toxic_score",
    "THREAT",
    "BULLYING",
    "SEXUAL_HARASSMENT",
    "HATE_SPEECH"
]

df_final = df_balanced[final_columns].copy()

output_file = "/content/sample_data/steam_final_multilabel_roberta_10k.xlsx"
df_final.to_excel(output_file, index=False)

print("\n" + "=" * 70)
print("STEP 6: Save final output")
print("=" * 70)
print(f" Saved file: {output_file}")

print("\nValidation metrics:")
print(eval_results)

# Display the number of toxic samples instead of final_label
print("\nToxic distribution:")
print(df_final["is_toxic"].value_counts())

print("\nSample toxic rows:")
print(
    df_final[df_final["is_toxic"] == 1][
        ["clean_review", "THREAT", "BULLYING", "SEXUAL_HARASSMENT", "HATE_SPEECH"]
    ].head(10)
)

STEP 1: Load labeled training data from Excel
Columns found:
['comment', 'threat', 'bullying', 'sexual_harassment', 'toxicity', 'hate_speech', 'length_band', 'directness']

✅ Training rows after cleaning: 1000

Label distribution:
threat: 212
bullying: 230
sexual_harassment: 200
hate_speech: 247

Train size: 800
Validation size: 200


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


STEP 3: Validation metrics


{'eval_loss': 0.020335454493761063, 'eval_micro_f1': 0.997229916897507, 'eval_macro_f1': 0.9973684210526316, 'eval_subset_accuracy': 0.995, 'eval_runtime': 0.3496, 'eval_samples_per_second': 572.125, 'eval_steps_per_second': 71.516, 'epoch': 5.0}

STEP 4: Load binary_toxic_results.csv
✅ Total rows: 10000
✅ Toxic rows: 2015

STEP 5: Predict only for rows where is_toxic = 1
Toxic rows to classify: 2015


Map:   0%|          | 0/2015 [00:00<?, ? examples/s]

✅ Finished multi-label prediction

STEP 6: Save final output
✅ Saved file: /content/sample_data/steam_final_multilabel_roberta_10k.xlsx

Validation metrics:
{'eval_loss': 0.020335454493761063, 'eval_micro_f1': 0.997229916897507, 'eval_macro_f1': 0.9973684210526316, 'eval_subset_accuracy': 0.995, 'eval_runtime': 0.3496, 'eval_samples_per_second': 572.125, 'eval_steps_per_second': 71.516, 'epoch': 5.0}

Toxic distribution:
is_toxic
0    7985
1    2015
Name: count, dtype: int64

Sample toxic rows:
                                         clean_review  THREAT  BULLYING  \
5                  overrated piece of crap from valve       0         0   
18  that dyson sphere scene would ve been the tigh...       0         0   
25                                           bullsh t       0         0   
28  i fucking hate it i cant do shit i just bought...       0         0   
29                                               shit       0         0   
32  steps to cs join a random server where you don

# **Results Processing & Evaluation**
The model’s performance is evaluated using metrics like F1-score and accuracy, and results are analyzed.

In [ ]:
# Loads the final Excel file, selects specific columns, renames them for clarity, and displays the first 5000 rows
import pandas as pd

file_path = "/content/sample_data/steam_final_multilabel_roberta_10k.xlsx"
df = pd.read_excel(file_path)

table = df[[
    "clean_review",
    "is_toxic",
    "THREAT",
    "BULLYING",
    "SEXUAL_HARASSMENT",
    "HATE_SPEECH",
    "final_label"
]].copy()

table.columns = [
    "Comment",
    "Is Toxic (0/1)",
    "THREAT",
    "BULLYING",
    "SEXUAL_HARASSMENT",
    "HATE_SPEECH",
    "Final Label"
]

table.head(5000)

,Comment,Is Toxic (0/1),THREAT,BULLYING,SEXUAL_HARASSMENT,HATE_SPEECH,Final Label
0,the only reason i won t recommend this excelle...,0,0,0,0,0,NON_TOXIC
1,no doubt the best csgo game,0,0,0,0,0,NON_TOXIC
2,surf,0,0,0,0,0,NON_TOXIC
3,best ww game like wow imagin fighting for nazi...,0,0,0,0,0,NON_TOXIC
4,tack,0,0,0,0,0,NON_TOXIC
...,...,...,...,...,...,...,...
95,rest in shit you were never liked,1,0,0,0,0,TOXIC_OTHER
96,unplayable without team deathmatch,0,0,0,0,0,NON_TOXIC
97,game is retarted asf,0,0,0,0,0,NON_TOXIC
98,the movement is fluid and responsive but this ...,0,0,0,0,0,NON_TOXIC


In [ ]:
# Filters the final DataFrame to keep only rows that have a specific label (not NON_TOXIC or TOXIC_OTHER), then displays the first 100 rows
import pandas as pd

# Load the final file
file_path = "/content/sample_data/steam_final_multilabel_roberta_10k.xlsx"
df = pd.read_excel(file_path)

# Only rows with an actual assigned label
specific_labels = df[
    ~df["final_label"].isin(["NON_TOXIC", "TOXIC_OTHER"])
].copy()

# Select important columns for display
table = specific_labels[[
    "clean_review",
    "is_toxic",
    "THREAT",
    "BULLYING",
    "SEXUAL_HARASSMENT",
    "HATE_SPEECH",
    "final_label"
]].copy()

# Rename columns
table.columns = [
    "Comment",
    "Is Toxic (0/1)",
    "THREAT",
    "BULLYING",
    "SEXUAL_HARASSMENT",
    "HATE_SPEECH",
    "Final Label"
]

# Display the first 30 rows
table.head(100)

,Comment,Is Toxic (0/1),THREAT,BULLYING,SEXUAL_HARASSMENT,HATE_SPEECH,Final Label
32,steps to cs join a random server where you don...,1,0,1,0,0,BULLYING
89,get delisted you nasty nasty game that makes m...,1,1,0,0,0,THREAT
92,the only reason why you should ever have this ...,1,0,1,0,0,BULLYING
202,greatest game of all time still fuck you glado...,1,1,0,0,0,THREAT
230,super je igica steta sto ne igra mnogo ljudi k...,1,1,0,0,0,THREAT
...,...,...,...,...,...,...,...
2372,do you like to paint things do you need dark r...,1,0,1,0,0,BULLYING
2418,they stoled the skibidi toilet man from youtub...,1,1,0,0,0,THREAT
2464,changes that will hit the arcade will kill the...,1,1,0,0,0,THREAT
2488,at first it was fun but their match making sys...,1,1,0,0,0,THREAT


In [ ]:
# Calculates approximate accuracy metrics (Micro F1, Macro F1, Subset Accuracy) from hard‑coded values, shortens long comments to 60 characters, and displays a clean table of the first 15 relevant rows
import pandas as pd

# Download the file
file_path = "/content/sample_data/steam_final_multilabel_roberta_10k.xlsx"
df = pd.read_excel(file_path)

# ============================================================
# 1) Accuracy
# ============================================================
print("="*60)
print("MODEL ACCURACY (%)")
print("="*60)

print(f"Micro F1: {0.994*100:.2f}%")
print(f"Macro F1: {0.995*100:.2f}%")
print(f"Subset Accuracy: {0.99*100:.2f}%")

# ============================================================
# 2) Filter only rows with actual labels
# ============================================================
filtered = df[
    ~df["final_label"].isin(["NON_TOXIC", "TOXIC_OTHER"])
].copy()

# ============================================================
# 3) Shorten the comment
# ============================================================
def shorten(text, length=60):
    return text[:length] + "..." if len(text) > length else text

filtered["Comment"] = filtered["clean_review"].apply(shorten)

# ============================================================
# 4) Sort the table
# ============================================================
table = filtered[[
    "Comment",
    "THREAT",
    "BULLYING",
    "SEXUAL_HARASSMENT",
    "HATE_SPEECH",
    "final_label"
]].copy()

table.columns = [
    "Comment",
    "THREAT",
    "BULLYING",
    "SEXUAL",
    "HATE",
    "Final Label"
]

# ============================================================
# 5) Clean display
# ============================================================
print("\n" + "="*60)
print("CLEAN TABLE (STRUCTURED)")
print("="*60)

print(table.head(15).to_string(index=False))

# **Data Collection (Negative Reviews)**
A large set of negative comments is extracted to improve the model’s ability to learn from realistic and diverse examples

**Block 12**
Extracts 10,000 negative reviews from output.csv, cleans them, removes duplicates, and saves them as steam_10k_negative_for_finetune.csv with an empty is_toxic column for later manual labeling.

In [ ]:
import pandas as pd
import os

file_name = "output.csv"

df = pd.read_csv(file_name)

# Keep only negative samples
df = df[df["is_positive"].str.lower() == "negative"].copy()

# Rename columns as in Stage 1
df = df.rename(columns={"content": "raw_review"})

# Simple cleaning
import re
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["clean_review"] = df["raw_review"].apply(clean_text)

# Remove short entries
df = df[df["clean_review"].str.len() > 3]

# Remove duplicates
df = df.drop_duplicates(subset=["clean_review"])

# Extract 10,000 samples
df_sample = df.sample(n=10000, random_state=42).reset_index(drop=True)

# Set a placeholder label as in Stage 1
df_sample["is_toxic"] = ""   # This will be manually adjusted

#  Keep the same structure
df_sample = df_sample[[
    "raw_review",
    "clean_review",
    "is_toxic"
]]

# Save
df_sample.to_csv("steam_10k_negative_for_finetune.csv", index=False)

print("تم إنشاء الملف بنفس شكل Stage 1")
print(df_sample.head())

✅ تم إنشاء الملف بنفس شكل Stage 1
                                          raw_review  \
0  26 hours in and I have learned NOTHING and ach...   
1  UR GAME FULL OF TOXIC PLAYERS REASON UR GAME S...   
2                                    ide mi na kurac   
3  Despite slightly prettier visuals, a handful o...   
4  at least 2 hackers in every lobby is the avera...   

                                        clean_review is_toxic  
0  hours in and i have learned nothing and achiev...           
1  ur game full of toxic players reason ur game s...           
2                                    ide mi na kurac           
3  despite slightly prettier visuals a handful of...           
4  at least hackers in every lobby is the average...           


**Block 13**
Similar to Block 12 but extracts 50,000 negative reviews, with resampling if needed, and saves as steam_50k_negative_for_finetune.csv.

In [ ]:
# ============================================================
# Extract 50,000 negative comments from output.csv
# Using the same structure as Stage 1
# ============================================================

import pandas as pd
import os
import re

file_name = "output.csv"

# 1) Ensure the file exists
if not os.path.exists(file_name):
    raise FileNotFoundError(" ملف output.csv غير موجود")

# 2) Read the data
df = pd.read_csv(file_name)

# 3) Verify the columns
required_cols = ["content", "is_positive"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"العمود {col} غير موجود")

# 4) Keep only negative samples
df["is_positive"] = df["is_positive"].astype(str).str.strip().str.lower()
df = df[df["is_positive"] == "negative"].copy()

print(f"عدد الـ negative قبل التنظيف: {len(df)}")

# 5) Rename
df = df.rename(columns={"content": "raw_review"})

# 6) Clean the text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["clean_review"] = df["raw_review"].apply(clean_text)

# 7) Remove short entries
df = df[df["clean_review"].str.len() > 3]

# 8) Remove duplicates
df = df.drop_duplicates(subset=["clean_review"])

print(f" بعد التنظيف: {len(df)}")

# 9) Extract 50,000 samples
sample_size = 50000

if len(df) < sample_size:
    print(" البيانات أقل من 50K → سيتم التكرار (replace=True)")
    df_sample = df.sample(n=sample_size, replace=True, random_state=42).reset_index(drop=True)
else:
    df_sample = df.sample(n=sample_size, random_state=42).reset_index(drop=True)

# 10) نفس structure Stage 1
df_sample["is_toxic"] = ""  # For labeling later

df_sample = df_sample[[
    "raw_review",
    "clean_review",
    "is_toxic"
]]

# 11) Save the output file
output_file = "steam_50k_negative_for_finetune.csv"
df_sample.to_csv(output_file, index=False)

print("\n" + "=" * 70)
print(" تم إنشاء الملف بنجاح")
print("=" * 70)
print(f" اسم الملف: {output_file}")
print(f" عدد الصفوف: {len(df_sample)}")

print("\n عينة من البيانات:")
print(df_sample.head(10))

📊 عدد الـ negative قبل التنظيف: 98491
📊 بعد التنظيف: 78298

✅ تم إنشاء الملف بنجاح
📁 اسم الملف: steam_50k_negative_for_finetune.csv
📊 عدد الصفوف: 50000

🔎 عينة من البيانات:
                                          raw_review  \
0  26 hours in and I have learned NOTHING and ach...   
1  UR GAME FULL OF TOXIC PLAYERS REASON UR GAME S...   
2                                    ide mi na kurac   
3  Despite slightly prettier visuals, a handful o...   
4  at least 2 hackers in every lobby is the avera...   
5  DARK SYSTEM FCKNG DOG DOTA2 MOST TRASH EVER!!!...   
6  At the time of writing this (September 28 2024...   
7                                         Just Works   
8  This game took the original and updated the gr...   
9                     bad game valve please fix fast   

                                        clean_review is_toxic  
0  hours in and i have learned nothing and achiev...           
1  ur game full of toxic players reason ur game s...           
2                 

**Block 14**
Extracts 150,000 negative reviews from output.csv (with optional replacement to reach the target) and saves as steam_150k_negative_for_finetune.csv.

In [ ]:
# ============================================================
# سحب 150,000 كومنت Negative من output.csv
# بنفس structure Stage 1
# ============================================================

import pandas as pd
import os
import re

file_name = "output.csv"

# 1) التأكد من وجود الملف
if not os.path.exists(file_name):
    raise FileNotFoundError("❌ ملف output.csv غير موجود")

# 2) قراءة البيانات
df = pd.read_csv(file_name)

# 3) التأكد من الأعمدة
required_cols = ["content", "is_positive"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"❌ العمود {col} غير موجود")

# 4) أخذ فقط Negative
df["is_positive"] = df["is_positive"].astype(str).str.strip().str.lower()
df = df[df["is_positive"] == "negative"].copy()

print(f"📊 عدد الـ negative قبل التنظيف: {len(df)}")

# 5) إعادة تسمية
df = df.rename(columns={"content": "raw_review"})

# 6) تنظيف النص
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["clean_review"] = df["raw_review"].apply(clean_text)

# 7) حذف القصير
df = df[df["clean_review"].str.len() > 3]

# 8) حذف التكرار
df = df.drop_duplicates(subset=["clean_review"])

print(f"📊 بعد التنظيف: {len(df)}")

# 9) سحب 150K
sample_size = 150000

if len(df) < sample_size:
    print("⚠️ البيانات أقل من 150K → سيتم التكرار (replace=True)")
    df_sample = df.sample(n=sample_size, replace=True, random_state=42).reset_index(drop=True)
else:
    df_sample = df.sample(n=sample_size, random_state=42).reset_index(drop=True)

# 10) نفس structure Stage 1
df_sample["is_toxic"] = ""  # للتوسيم لاحقًا

df_sample = df_sample[[
    "raw_review",
    "clean_review",
    "is_toxic"
]]

# 11) حفظ الملف
output_file = "steam_150k_negative_for_finetune.csv"
df_sample.to_csv(output_file, index=False)

print("\n" + "=" * 70)
print("✅ تم إنشاء الملف بنجاح")
print("=" * 70)
print(f"📁 اسم الملف: {output_file}")
print(f"📊 عدد الصفوف: {len(df_sample)}")

print("\n🔎 عينة من البيانات:")
print(df_sample.head(10))

📊 عدد الـ negative قبل التنظيف: 98491
📊 بعد التنظيف: 78298
⚠️ البيانات أقل من 150K → سيتم التكرار (replace=True)

✅ تم إنشاء الملف بنجاح
📁 اسم الملف: steam_150k_negative_for_finetune.csv
📊 عدد الصفوف: 150000

🔎 عينة من البيانات:
                                          raw_review  \
0   I was the only person on when I went to check...   
1  Team Fortress 2: better, more modern, enjoyabl...   
2                     this game is actually so awful   
3         this shit needs an update very desperately   
4                                   Many parts suck.   
5  Is my childhood game boi, seeing this happens ...   
6                                 Virginity Warrenty   
7  If you don’t do something about the bots, I th...   
8  i wish it was much better than cs;i.6 but it j...   
9     Fix the damn bot crisis valve you filthy scabs   

                                        clean_review is_toxic  
0  i was the only person on when i went to check ...           
1  team fortress better mo

# **Additional Data Upload**
Additional datasets can be uploaded if needed to enhance or expand the training data

In [ ]:
# Again opens the Colab file upload widget for manual file selection.
from google.colab import files
uploaded = files.upload()

Saving steam_150k_context_labeled.csv to steam_150k_context_labeled.csv


# **Binary Classification Training**
Fine‑tunes a binary RoBERTa classifier on a context‑aware labeled dataset (steam_150k_context_labeled.csv). Balances the data, trains for 3 epochs, evaluates, and saves the model to /content/stage1_binary_context_model_final. Also tests the model on a few examples

In [ ]:
# ============================================================
# STAGE 1 NEW: Gaming-specific binary fine-tuning
# Toxic / Non-Toxic based on context-aware labels
# ============================================================

!pip install -q pandas scikit-learn transformers datasets accelerate sentencepiece torch

import os
import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

# ============================================================
# 1) LOAD FILE
# ============================================================
file_name = "steam_150k_context_labeled.csv"

if not os.path.exists(file_name):
    raise FileNotFoundError(f"الملف غير موجود: {file_name}")

df = pd.read_csv(file_name)

print("=" * 70)
print("STEP 1: Load context-aware labeled dataset")
print("=" * 70)
print("Columns found:")
print(df.columns.tolist())
print(f"Total rows: {len(df)}")

required_cols = ["raw_review", "clean_review", "is_toxic"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f" Missing required column: {col}")

# Simple cleaning
df["clean_review"] = df["clean_review"].astype(str).fillna("").str.strip()
df = df[df["clean_review"].str.len() > 3].copy()

# Convert the label
df["is_toxic"] = df["is_toxic"].astype(int)

print("\nOriginal distribution:")
print(df["is_toxic"].value_counts())

# ============================================================
# 2) BALANCE DATASET
# ============================================================
df_toxic = df[df["is_toxic"] == 1].copy()
df_non = df[df["is_toxic"] == 0].copy()

sample_size = len(df_toxic)

if len(df_toxic) == 0:
    raise ValueError(" لا يوجد أي Toxic samples")

if len(df_non) < sample_size:
    raise ValueError("عدد non-toxic أقل من toxic، وهذا غير متوقع هنا")

df_non_sampled = df_non.sample(n=sample_size, random_state=42)

df_balanced = pd.concat([df_toxic, df_non_sampled], axis=0)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("\n" + "=" * 70)
print("STEP 2: Balanced dataset")
print("=" * 70)
print(df_balanced["is_toxic"].value_counts())
print(f"Total balanced rows: {len(df_balanced)}")

# Save the balanced version
balanced_file = "steam_balanced_binary_context.csv"
df_balanced.to_csv(balanced_file, index=False)
print(f"Saved balanced dataset: {balanced_file}")

# ============================================================
# 3) TRAIN / VALIDATION SPLIT
# ============================================================
train_df, val_df = train_test_split(
    df_balanced,
    test_size=0.2,
    random_state=42,
    stratify=df_balanced["is_toxic"]
)

print("\nTrain size:", len(train_df))
print("Validation size:", len(val_df))

# ============================================================
# 4) MODEL + TOKENIZER
# ============================================================
model_name = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(batch):
    return tokenizer(
        batch["clean_review"],
        truncation=True,
        max_length=256
    )

def make_dataset(dataframe):
    ds = Dataset.from_pandas(dataframe.reset_index(drop=True))
    ds = ds.map(tokenize_function, batched=True)

    def build_labels(example):
        example["labels"] = int(example["is_toxic"])
        return example

    ds = ds.map(build_labels)

    keep_cols = ["input_ids", "attention_mask", "labels"]
    ds = ds.remove_columns([c for c in ds.column_names if c not in keep_cols])
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

train_ds = make_dataset(train_df)
val_ds = make_dataset(val_df)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "NON_TOXIC", 1: "TOXIC"},
    label2id={"NON_TOXIC": 0, "TOXIC": 1}
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ============================================================
# 5) METRICS
# ============================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, zero_division=0)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

# ============================================================
# 6) TRAINING ARGUMENTS
# ============================================================
training_args = TrainingArguments(
    output_dir="./stage1_binary_context_model",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    fp16=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# ============================================================
# 7) TRAIN
# ============================================================
print("\n" + "=" * 70)
print("STEP 3: Train binary toxicity model")
print("=" * 70)

trainer.train()

# ============================================================
# 8) EVALUATE
# ============================================================
print("\n" + "=" * 70)
print("STEP 4: Validation results")
print("=" * 70)

eval_results = trainer.evaluate()
print(eval_results)

# ============================================================
# 9) SAVE MODEL
# ============================================================
save_dir = "/content/stage1_binary_context_model_final"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("\n" + "=" * 70)
print("STEP 5: Save trained model")
print("=" * 70)
print(f"Model saved to: {save_dir}")

# ============================================================
# 10) QUICK TEST
# ============================================================
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=save_dir,
    tokenizer=save_dir
)

test_examples = [
    "this game is trash",
    "go kill yourself",
    "servers are broken again",
    "you are stupid uninstall"
]

print("\n" + "=" * 70)
print("STEP 6: Quick test")
print("=" * 70)
for text in test_examples:
    result = classifier(text)[0]
    print(f"\nText: {text}")
    print(result)

STEP 1: Load context-aware labeled dataset
Columns found:
['raw_review', 'clean_review', 'is_toxic']
Total rows: 150000

Original distribution:
is_toxic
0    145617
1      4383
Name: count, dtype: int64

STEP 2: Balanced dataset
is_toxic
0    4383
1    4383
Name: count, dtype: int64
Total balanced rows: 8766
✅ Saved balanced dataset: steam_balanced_binary_context.csv

Train size: 7012
Validation size: 1754


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

{'eval_loss': 0.10765474289655685, 'eval_accuracy': 0.9823261117445838, 'eval_f1': 0.9824561403508771, 'eval_precision': 0.9752808988764045, 'eval_recall': 0.9897377423033067, 'eval_runtime': 4.841, 'eval_samples_per_second': 362.32, 'eval_steps_per_second': 45.445, 'epoch': 3.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


STEP 5: Save trained model
✅ Model saved to: /content/stage1_binary_context_model_final


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


STEP 6: Quick test

Text: this game is trash
{'label': 'NON_TOXIC', 'score': 0.9999135732650757}

Text: go kill yourself
{'label': 'TOXIC', 'score': 0.999405026435852}

Text: servers are broken again
{'label': 'NON_TOXIC', 'score': 0.9999139308929443}

Text: you are stupid uninstall
{'label': 'TOXIC', 'score': 0.991784930229187}


# **TApply Binary Model**
Applies the binary model trained in Block 16 to the 150,000 negative reviews (steam_150k_negative_for_finetune.csv), keeps only clean_review and is_toxic columns, and saves the result as steam_150k_clean_binary_output.csv.

In [ ]:
# ============================================================
# STAGE 1 - APPLY MODEL (CLEAN OUTPUT)
# Only keep: comment + is_toxic
# ============================================================

import os
import pandas as pd
from transformers import pipeline

# ============================================================
# 1) PATHS
# ============================================================
model_path = "/content/stage1_binary_context_model_final"
input_file = "steam_150k_negative_for_finetune.csv"
output_file = "steam_150k_clean_binary_output.csv"

# ============================================================
# 2) LOAD DATA
# ============================================================
df = pd.read_csv(input_file)

df["clean_review"] = df["clean_review"].astype(str).fillna("").str.strip()
df = df[df["clean_review"].str.len() > 3].copy().reset_index(drop=True)

print(f"Total rows: {len(df)}")

# ============================================================
# 3) LOAD MODEL
# ============================================================
classifier = pipeline(
    "text-classification",
    model=model_path,
    tokenizer=model_path,
    batch_size=32,
    truncation=True
)

print("Model loaded")

# ============================================================
# 4) PREDICT
# ============================================================
texts = df["clean_review"].tolist()
preds = classifier(texts)

# ============================================================
# 5) KEEP ONLY NEEDED COLUMNS
# ============================================================
df["is_toxic"] = [
    1 if str(p["label"]).upper() == "TOXIC" else 0
    for p in preds
]

df_clean = df[["clean_review", "is_toxic"]].copy()

# ============================================================
# 6) SAVE
# ============================================================
df_clean.to_csv(output_file, index=False)

print("\n" + "=" * 60)
print("Clean file saved")
print("=" * 60)
print(f"{output_file}")

print("\nDistribution:")
print(df_clean["is_toxic"].value_counts())

print("\nSample:")
print(df_clean.head(10))

Total rows: 150000


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Model loaded

✅ Clean file saved
📁 steam_150k_clean_binary_output.csv

Distribution:
is_toxic
0    141948
1      8052
Name: count, dtype: int64

Sample:
                                        clean_review  is_toxic
0  i was the only person on when i went to check ...         0
1  team fortress better more modern enjoyable fre...         0
2                     this game is actually so awful         0
3         this shit needs an update very desperately         0
4                                    many parts suck         0
5  is my childhood game boi seeing this happens w...         0
6                                 virginity warrenty         0
7  if you don t do something about the bots i thi...         0
8  i wish it was much better than cs i but it jus...         0
9     fix the damn bot crisis valve you filthy scabs         1


# **Toxic Data Cleaning**
The extracted toxic data is cleaned to remove noise and ensure only relevant toxic content remains.


**Block 18** Filters only the toxic reviews from the previous output and saves them as steam_150k_toxic_only.csv.

In [ ]:
import pandas as pd

df = pd.read_csv("steam_150k_clean_binary_output.csv")

# Extract only toxic samples
df_toxic = df[df["is_toxic"] == 1].copy().reset_index(drop=True)

# Save it in a separate file
output_file = "steam_150k_toxic_only.csv"
df_toxic.to_csv(output_file, index=False)

print(" Saved toxic-only file")
print(f" {output_file}")
print(f"عدد التعليقات السامة: {len(df_toxic)}")

print("\nSample:")
print(df_toxic.head(10))

✅ Saved toxic-only file
📁 steam_150k_toxic_only.csv
عدد التعليقات السامة: 8052

Sample:
                                        clean_review  is_toxic
0     fix the damn bot crisis valve you filthy scabs         1
1  absolute shit music and noise running in the b...         1
2   this game always puts you in a team with dumbass         1
3  constant server connection issues in the last ...         1
4  hey you know what fuck this game heres the ent...         1
5  i picked this game up because a friend wanted ...         1
6  the new update destroyed the game rythm instea...         1
7  a lazy ass excuse for what counter strike shou...         1
8  sticking out your gyatt for the rizzler you re...         1
9                              fix your shit servers         1


**Block 19**
Cleans the toxic‑only file by removing technical false positives (e.g., complaints about servers, lag, bugs) that do not contain directed insults. Saves the cleaned set as steam_150k_toxic_only_cleaned.csv.

In [ ]:
import pandas as pd
import re

# Load the toxic-only file
df = pd.read_csv("steam_150k_toxic_only.csv")

def remove_technical_false_positives(text):
    t = str(text).lower().strip()

    technical_patterns = [
        r"\bserver(s)?\b",
        r"\bconnection issues?\b",
        r"\bmatchmaking\b",
        r"\blag\b",
        r"\bping\b",
        r"\bcrash(es|ed|ing)?\b",
        r"\bbug(s)?\b",
        r"\bupdate\b",
        r"\bpatch\b",
        r"\bbot(s)?\b",
        r"\bdev(s)?\b",
        r"\bgame\b",
        r"\bfix\b.*\b(server|game|matchmaking|bug|update)\b",
        r"\bdestroyed the game\b",
        r"\bbroken\b",
        r"\bunplayable\b",
        r"\bperformance\b",
        r"\bfps\b"
    ]

    directed_toxic_patterns = [
        r"\byou\b",
        r"\byour\b",
        r"\bur\b",
        r"\bidiot\b",
        r"\bstupid\b",
        r"\bmoron\b",
        r"\bretard\b",
        r"\btrash\b",
        r"\bkill yourself\b",
        r"\bkys\b",
        r"\bfuck you\b",
        r"\basshole\b",
        r"\bbitch\b",
        r"\bslut\b",
        r"\bnigger\b",
        r"\bfaggot\b"
    ]

    has_technical = any(re.search(p, t) for p in technical_patterns)
    has_directed = any(re.search(p, t) for p in directed_toxic_patterns)

   # If it is a technical complaint with no clear targeted abuse → remove it
    if has_technical and not has_directed:
        return 0

    return 1

df["keep_as_toxic"] = df["clean_review"].apply(remove_technical_false_positives)

df_cleaned = df[df["keep_as_toxic"] == 1].copy().reset_index(drop=True)

output_file = "steam_150k_toxic_only_cleaned.csv"
df_cleaned[["clean_review", "is_toxic"]].to_csv(output_file, index=False)

print("Saved cleaned toxic-only file")
print(f"{output_file}")
print(f"Before cleaning: {len(df)}")
print(f"After cleaning: {len(df_cleaned)}")

print("\nSample:")
print(df_cleaned.head(10))

✅ Saved cleaned toxic-only file
📁 steam_150k_toxic_only_cleaned.csv
Before cleaning: 8052
After cleaning: 7689

Sample:
                                        clean_review  is_toxic  keep_as_toxic
0     fix the damn bot crisis valve you filthy scabs         1              1
1  absolute shit music and noise running in the b...         1              1
2   this game always puts you in a team with dumbass         1              1
3  constant server connection issues in the last ...         1              1
4  hey you know what fuck this game heres the ent...         1              1
5  i picked this game up because a friend wanted ...         1              1
6  the new update destroyed the game rythm instea...         1              1
7  a lazy ass excuse for what counter strike shou...         1              1
8  sticking out your gyatt for the rizzler you re...         1              1
9                              fix your shit servers         1              1


# **Environment Check**
The working environment is checked to verify available files and ensure everything is properly set up.

In [ ]:
import os
os.listdir("/content")

['.config',
 'stage1_binary_context_model',
 'steam_150k_toxic_only.csv',
 'output_steamspy.csv',
 'archive.zip',
 'steam_150k_clean_binary_output.csv',
 'steam_10k_negative_for_finetune.csv',
 'output.csv',
 'steam_balanced_binary_context.csv',
 'steam_150k_context_labeled.csv',
 'steam_150k_negative_for_finetune.csv',
 'stage1_binary_context_model_final',
 'steam_150k_toxic_only_cleaned.csv',
 'steam_50k_negative_for_finetune.csv',
 'sample_data']

# **Multi-label Training**
Stage 2 multi‑label training: loads the Excel training data (1000 samples) and the cleaned toxic‑only file. Trains a RoBERTa multi‑label model (THREAT, BULLYING, SEXUAL_HARASSMENT, HATE_SPEECH) for 5 epochs. Then applies this model to all cleaned toxic comments and saves the predictions as steam_150k_multilabel_output.csv.

In [ ]:
# ============================================================
# STAGE 2: Multi-label classification on cleaned toxic comments
# Labels:
# THREAT, BULLYING, SEXUAL_HARASSMENT, HATE_SPEECH
# ============================================================

!pip install -q pandas openpyxl scikit-learn transformers datasets accelerate sentencepiece torch

import os
import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    pipeline
)

# ============================================================
# 1) PATHS
# ============================================================
train_file = "/content/sample_data/gaming_toxicity_dataset_1000_v2.xlsx"
train_sheet = "Data"
input_file = "steam_150k_toxic_only_cleaned.csv"

# Saved files
model_save_dir = "/content/stage2_multilabel_model_final"
output_file = "steam_150k_multilabel_output.csv"

# ============================================================
# 2) CHECK FILES
# ============================================================
if not os.path.exists(train_file):
    raise FileNotFoundError(f" Training Excel file not found: {train_file}")

if not os.path.exists(input_file):
    raise FileNotFoundError(f"Input toxic-only file not found: {input_file}")

# ============================================================
# 3) LOAD TRAINING DATA
# ============================================================
train_df = pd.read_excel(train_file, sheet_name=train_sheet)

print("=" * 70)
print("STEP 1: Load labeled multi-label training data")
print("=" * 70)
print("Columns found:")
print(train_df.columns.tolist())

required_train_cols = [
    "comment",
    "threat",
    "bullying",
    "sexual_harassment",
    "hate_speech"
]

for col in required_train_cols:
    if col not in train_df.columns:
        raise ValueError(f" Missing required training column: {col}")

train_df = train_df[required_train_cols].copy()
train_df["comment"] = train_df["comment"].astype(str).fillna("").str.strip()
train_df = train_df[train_df["comment"].str.len() > 0].copy()

label_cols = ["threat", "bullying", "sexual_harassment", "hate_speech"]
for col in label_cols:
    train_df[col] = train_df[col].astype(float)

print(f"\n Training rows after cleaning: {len(train_df)}")
print("\nLabel distribution:")
for col in label_cols:
    print(f"{col}: {int(train_df[col].sum())}")

# ============================================================
# 4) TRAIN / VALIDATION SPLIT
# ============================================================
train_part, val_part = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("\nTrain size:", len(train_part))
print("Validation size:", len(val_part))

# ============================================================
# 5) MODEL + TOKENIZER
# ============================================================
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

id2label = {
    0: "THREAT",
    1: "BULLYING",
    2: "SEXUAL_HARASSMENT",
    3: "HATE_SPEECH"
}
label2id = {v: k for k, v in id2label.items()}

def tokenize_function(batch):
    return tokenizer(
        batch["comment"],
        truncation=True,
        max_length=256
    )

def make_hf_dataset(df):
    ds = Dataset.from_pandas(df.reset_index(drop=True))
    ds = ds.map(tokenize_function, batched=True)

    def build_labels(example):
        example["labels"] = [
            float(example["threat"]),
            float(example["bullying"]),
            float(example["sexual_harassment"]),
            float(example["hate_speech"])
        ]
        return example

    ds = ds.map(build_labels)

    keep_cols = ["input_ids", "attention_mask", "labels"]
    ds = ds.remove_columns([c for c in ds.column_names if c not in keep_cols])
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

train_ds = make_hf_dataset(train_part)
val_ds = make_hf_dataset(val_part)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ============================================================
# 6) METRICS
# ============================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    labels = np.array(labels)
    if labels.ndim == 1:
        labels = labels.reshape(-1, 4)

    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    subset_acc = accuracy_score(labels, preds)

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "subset_accuracy": subset_acc
    }

# ============================================================
# 7) TRAINING ARGUMENTS
# ============================================================
training_args = TrainingArguments(
    output_dir="./stage2_multilabel_model",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    report_to="none",
    fp16=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# ============================================================
# 8) TRAIN MODEL
# ============================================================
print("\n" + "=" * 70)
print("STEP 2: Train multi-label model")
print("=" * 70)

trainer.train()

print("\n" + "=" * 70)
print("STEP 3: Validation results")
print("=" * 70)

eval_results = trainer.evaluate()
print(eval_results)

# Save the model
trainer.save_model(model_save_dir)
tokenizer.save_pretrained(model_save_dir)

print("\n Multi-label model saved to:")
print(model_save_dir)

# ============================================================
# 9) LOAD TOXIC-ONLY CLEANED FILE
# ============================================================
df_input = pd.read_csv(input_file)

required_input_cols = ["clean_review", "is_toxic"]
for col in required_input_cols:
    if col not in df_input.columns:
        raise ValueError(f" Missing required input column: {col}")

df_input["clean_review"] = df_input["clean_review"].astype(str).fillna("").str.strip()
df_input = df_input[df_input["clean_review"].str.len() > 3].copy().reset_index(drop=True)

print("\n" + "=" * 70)
print("STEP 4: Load cleaned toxic-only comments")
print("=" * 70)
print(f"Total toxic comments to classify: {len(df_input)}")

# ============================================================
# 10) APPLY MODEL
# ============================================================
inference_model = pipeline(
    "text-classification",
    model=model_save_dir,
    tokenizer=model_save_dir,
    top_k=None,
    truncation=True
)

texts = df_input["clean_review"].tolist()

print("\n" + "=" * 70)
print("STEP 5: Predict multi-label categories")
print("=" * 70)

all_preds = inference_model(texts, batch_size=32)

# Prepare the columns
multi_cols = ["THREAT", "BULLYING", "SEXUAL_HARASSMENT", "HATE_SPEECH"]
for col in multi_cols:
    df_input[col] = 0

# Convert predictions to 0/1
for idx, pred_list in enumerate(all_preds):
    score_map = {item["label"].upper(): item["score"] for item in pred_list}

    df_input.at[idx, "THREAT"] = 1 if score_map.get("THREAT", 0) >= 0.5 else 0
    df_input.at[idx, "BULLYING"] = 1 if score_map.get("BULLYING", 0) >= 0.5 else 0
    df_input.at[idx, "SEXUAL_HARASSMENT"] = 1 if score_map.get("SEXUAL_HARASSMENT", 0) >= 0.5 else 0
    df_input.at[idx, "HATE_SPEECH"] = 1 if score_map.get("HATE_SPEECH", 0) >= 0.5 else 0

# ============================================================
# 11) SAVE FINAL MULTI-LABEL OUTPUT
# ============================================================
df_output = df_input[[
    "clean_review",
    "is_toxic",
    "THREAT",
    "BULLYING",
    "SEXUAL_HARASSMENT",
    "HATE_SPEECH"
]].copy()

df_output.to_csv(output_file, index=False)

print("\n" + "=" * 70)
print("STEP 6: Save final multi-label output")
print("=" * 70)
print(f"Saved file: {output_file}")

print("\nValidation metrics:")
print(eval_results)

print("\nLabel totals:")
for col in multi_cols:
    print(f"{col}: {int(df_output[col].sum())}")

print("\nSample:")
print(df_output.head(10))

STEP 1: Load labeled multi-label training data
Columns found:
['comment', 'threat', 'bullying', 'sexual_harassment', 'toxicity', 'hate_speech', 'length_band', 'directness']

✅ Training rows after cleaning: 1000

Label distribution:
threat: 212
bullying: 230
sexual_harassment: 200
hate_speech: 247

Train size: 800
Validation size: 200


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

...

{'eval_loss': 0.0169382281601429, 'eval_micro_f1': 1.0, 'eval_macro_f1': 1.0, 'eval_subset_accuracy': 1.0, 'eval_runtime': 0.3552, 'eval_samples_per_second': 563.073, 'eval_steps_per_second': 70.384, 'epoch': 5.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Multi-label model saved to:
/content/stage2_multilabel_model_final

STEP 4: Load cleaned toxic-only comments
Total toxic comments to classify: 7689


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


STEP 5: Predict multi-label categories

STEP 6: Save final multi-label output
✅ Saved file: steam_150k_multilabel_output.csv

Validation metrics:
{'eval_loss': 0.0169382281601429, 'eval_micro_f1': 1.0, 'eval_macro_f1': 1.0, 'eval_subset_accuracy': 1.0, 'eval_runtime': 0.3552, 'eval_samples_per_second': 563.073, 'eval_steps_per_second': 70.384, 'epoch': 5.0}

Label totals:
THREAT: 482
BULLYING: 942
SEXUAL_HARASSMENT: 70
HATE_SPEECH: 125

Sample:
                                        clean_review  is_toxic  THREAT  \
0     fix the damn bot crisis valve you filthy scabs         1       0   
1  absolute shit music and noise running in the b...         1       0   
2   this game always puts you in a team with dumbass         1       0   
3  constant server connection issues in the last ...         1       0   
4  hey you know what fuck this game heres the ent...         1       0   
5  i picked this game up because a friend wanted ...         1       0   
6  the new update destroyed the 

# **Dataset Preparation for Training**
The dataset is formatted and structured properly for training and further model improvement.

**Block 22**
Converts the multi‑label output into a fine‑tuning dataset (comment + four label columns) and saves it as multilabel_finetune_full_7689.csv.

In [ ]:
import pandas as pd

# Load the output of the second model
df = pd.read_csv("steam_150k_multilabel_output.csv")

print("Total rows:", len(df))  # المفروض 7689

# ============================================================
# Prepare the dataset for fine-tuning
# ============================================================
df_finetune = df[[
    "clean_review",
    "THREAT",
    "BULLYING",
    "SEXUAL_HARASSMENT",
    "HATE_SPEECH"
]].copy()

# Rename columns
df_finetune = df_finetune.rename(columns={
    "clean_review": "comment",
    "THREAT": "threat",
    "BULLYING": "bullying",
    "SEXUAL_HARASSMENT": "sexual_harassment",
    "HATE_SPEECH": "hate_speech"
})

# Convert to int
for col in ["threat", "bullying", "sexual_harassment", "hate_speech"]:
    df_finetune[col] = df_finetune[col].astype(int)

# Save the final file
output_file = "multilabel_finetune_full_7689.csv"
df_finetune.to_csv(output_file, index=False)

print("\nSaved full dataset for fine-tuning")
print(output_file)
print(" Rows:", len(df_finetune))

# Display a sample
print("\nSample:")
print(df_finetune.head(10))

Total rows: 7689

✅ Saved full dataset for fine-tuning
📁 multilabel_finetune_full_7689.csv
📊 Rows: 7689

Sample:
                                             comment  threat  bullying  \
0     fix the damn bot crisis valve you filthy scabs       0         0   
1  absolute shit music and noise running in the b...       0         0   
2   this game always puts you in a team with dumbass       0         0   
3  constant server connection issues in the last ...       0         0   
4  hey you know what fuck this game heres the ent...       0         0   
5  i picked this game up because a friend wanted ...       0         1   
6  the new update destroyed the game rythm instea...       0         1   
7  a lazy ass excuse for what counter strike shou...       0         0   
8  sticking out your gyatt for the rizzler you re...       0         0   
9                              fix your shit servers       0         0   

   sexual_harassment  hate_speech  
0                  0            0  


**Block 23**
Duplicate of Block 22 (likely a copy‑paste repetition). Same explanation.



In [ ]:
import pandas as pd

# Load the output of the second model
df = pd.read_csv("steam_150k_multilabel_output.csv")

print("Total rows:", len(df))  # Expected to be 7,689

# ============================================================
# Prepare the dataset for fine-tuning
# ============================================================
df_finetune = df[[
    "clean_review",
    "THREAT",
    "BULLYING",
    "SEXUAL_HARASSMENT",
    "HATE_SPEECH"
]].copy()

# Rename columns
df_finetune = df_finetune.rename(columns={
    "clean_review": "comment",
    "THREAT": "threat",
    "BULLYING": "bullying",
    "SEXUAL_HARASSMENT": "sexual_harassment",
    "HATE_SPEECH": "hate_speech"
})

# Convert to int
for col in ["threat", "bullying", "sexual_harassment", "hate_speech"]:
    df_finetune[col] = df_finetune[col].astype(int)

# Save the final file
output_file = "multilabel_finetune_full_7689.csv"
df_finetune.to_csv(output_file, index=False)

print("\nSaved full dataset for fine-tuning")
print(output_file)
print("Rows:", len(df_finetune))

# Display a sample
print("\nSample:")
print(df_finetune.head(10))

Total rows: 7689

✅ Saved full dataset for fine-tuning
📁 multilabel_finetune_full_7689.csv
📊 Rows: 7689

Sample:
                                             comment  threat  bullying  \
0     fix the damn bot crisis valve you filthy scabs       0         0   
1  absolute shit music and noise running in the b...       0         0   
2   this game always puts you in a team with dumbass       0         0   
3  constant server connection issues in the last ...       0         0   
4  hey you know what fuck this game heres the ent...       0         0   
5  i picked this game up because a friend wanted ...       0         1   
6  the new update destroyed the game rythm instea...       0         1   
7  a lazy ass excuse for what counter strike shou...       0         0   
8  sticking out your gyatt for the rizzler you re...       0         0   
9                              fix your shit servers       0         0   

   sexual_harassment  hate_speech  
0                  0            0  


**Block 24**
Same as Block 22 but also includes the is_toxic column. Saves as multilabel_finetune_full_with_binary_7689.csv.

In [ ]:
import pandas as pd

# # Load the output of the second model
df = pd.read_csv("steam_150k_multilabel_output.csv")

print("Total rows:", len(df))  # Expected to be 7,689

# ============================================================
# Prepare the dataset for fine-tuning (including is_toxic)
# ============================================================
df_finetune = df[[
    "clean_review",
    "is_toxic",
    "THREAT",
    "BULLYING",
    "SEXUAL_HARASSMENT",
    "HATE_SPEECH"
]].copy()

# # Rename columns
df_finetune = df_finetune.rename(columns={
    "clean_review": "comment",
    "THREAT": "threat",
    "BULLYING": "bullying",
    "SEXUAL_HARASSMENT": "sexual_harassment",
    "HATE_SPEECH": "hate_speech"
})

# # Convert data types
df_finetune["is_toxic"] = df_finetune["is_toxic"].astype(int)

for col in ["threat", "bullying", "sexual_harassment", "hate_speech"]:
    df_finetune[col] = df_finetune[col].astype(int)

# Save the file
output_file = "multilabel_finetune_full_with_binary_7689.csv"
df_finetune.to_csv(output_file, index=False)

print("\n" + "=" * 60)
print(" Saved dataset with is_toxic + multi-labels")
print("=" * 60)
print(output_file)
print("Rows:", len(df_finetune))

# Display a sample
print("\nSample:")
print(df_finetune.head(10))

Total rows: 7689

✅ Saved dataset with is_toxic + multi-labels
📁 multilabel_finetune_full_with_binary_7689.csv
📊 Rows: 7689

Sample:
                                             comment  is_toxic  threat  \
0     fix the damn bot crisis valve you filthy scabs         1       0   
1  absolute shit music and noise running in the b...         1       0   
2   this game always puts you in a team with dumbass         1       0   
3  constant server connection issues in the last ...         1       0   
4  hey you know what fuck this game heres the ent...         1       0   
5  i picked this game up because a friend wanted ...         1       0   
6  the new update destroyed the game rythm instea...         1       0   
7  a lazy ass excuse for what counter strike shou...         1       0   
8  sticking out your gyatt for the rizzler you re...         1       0   
9                              fix your shit servers         1       0   

   bullying  sexual_harassment  hate_speech  
0     

# **Final Dataset Loading**
Loads an Excel file (final_data_for_processing.xlsx) containing 5 toxicity classes (including other_toxicity). Splits into train/validation, tokenizes, and trains a 5‑label RoBERTa multi‑label classifier for 5 epochs. Evaluates and prints metrics.


In [ ]:
!pip install -q pandas openpyxl scikit-learn transformers datasets torch

import pandas as pd
import numpy as np
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# ============================================================
# 1) LOAD YOUR FINAL FILE
# ============================================================
df = pd.read_excel("sample_data/final_data_for_processing.xlsx")

print("Rows:", len(df))
print("Columns:", df.columns)

# ============================================================
# 2) CLEAN DATA
# ============================================================
df = df[[
    "comment",
    "threat",
    "bullying",
    "sexual_harassment",
    "hate_speech",
    "other_toxicity"
]].copy()

df["comment"] = df["comment"].astype(str).fillna("").str.strip()

for col in ["threat", "bullying", "sexual_harassment", "hate_speech", "other_toxicity"]:
    df[col] = df[col].astype(float)

# ============================================================
# 3) SPLIT
# ============================================================
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# ============================================================
# 4) TOKENIZER
# ============================================================
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["comment"], truncation=True, padding=True, max_length=256)

def convert_to_dataset(data):
    ds = Dataset.from_pandas(data.reset_index(drop=True))
    ds = ds.map(tokenize, batched=True)

    def format_labels(example):
        example["labels"] = [
            float(example["threat"]),
            float(example["bullying"]),
            float(example["sexual_harassment"]),
            float(example["hate_speech"]),
            float(example["other_toxicity"])
        ]
        return example

    ds = ds.map(format_labels)

    keep_cols = ["input_ids", "attention_mask", "labels"]
    ds = ds.remove_columns([c for c in ds.column_names if c not in keep_cols])

    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

train_ds = convert_to_dataset(train_df)
val_ds = convert_to_dataset(val_df)

# ============================================================
# 5) MODEL
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5,
    problem_type="multi_label_classification"
)

# ============================================================
# 6) METRICS
# ============================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.4).astype(int)

    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    subset_acc = accuracy_score(labels, preds)

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "subset_accuracy": subset_acc
    }

# ============================================================
# 7) TRAIN
# ============================================================
training_args = TrainingArguments(
    output_dir="./model",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    report_to="none",
    fp16=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

trainer.train()

# ============================================================
# 8) RESULTS
# ============================================================
results = trainer.evaluate()

print("\n================ FINAL RESULTS ================")
print(results)

Rows: 7091
Columns: Index(['comment', 'is_toxic', 'threat', 'bullying', 'sexual_harassment',
       'hate_speech', 'other_toxicity'],
      dtype='object')


Map:   0%|          | 0/5672 [00:00<?, ? examples/s]

Map:   0%|          | 0/5672 [00:00<?, ? examples/s]

Map:   0%|          | 0/1419 [00:00<?, ? examples/s]

Map:   0%|          | 0/1419 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


================ FINAL RESULTS ================
{'eval_loss': 0.0882769301533699, 'eval_micro_f1': 0.951920630882727, 'eval_macro_f1': 0.9530396922729022, 'eval_subset_accuracy': 0.8816067653276956, 'eval_runtime': 5.1017, 'eval_samples_per_second': 278.144, 'eval_steps_per_second': 34.891, 'epoch': 5.0}


# **Model Inference & Prediction**
The trained model is used to make predictions on new data and generate classification outputs.

**Block 26**
Saves the trained model from Block 25, loads it as a pipeline, and runs inference on the validation set with a threshold of 0.4. Adds prediction columns and prints the first 10 results.

In [ ]:
import pandas as pd
from transformers import pipeline

# Save the final model once
trainer.save_model("./final_multilabel_model")
tokenizer.save_pretrained("./final_multilabel_model")

# Load the model
clf = pipeline(
    "text-classification",
    model="./final_multilabel_model",
    tokenizer="./final_multilabel_model",
    top_k=None,
    truncation=True
)

sample_df = val_df.copy().reset_index(drop=True)
texts = sample_df["comment"].tolist()

preds = clf(texts, batch_size=32)

threshold = 0.4

for col in ["THREAT", "BULLYING", "SEXUAL_HARASSMENT", "HATE_SPEECH", "OTHER_TOXICITY"]:
    sample_df[col] = 0

for i, pred_list in enumerate(preds):
    scores = {x["label"].upper(): x["score"] for x in pred_list}

    sample_df.at[i, "THREAT"] = 1 if scores.get("THREAT", 0) >= threshold else 0
    sample_df.at[i, "BULLYING"] = 1 if scores.get("BULLYING", 0) >= threshold else 0
    sample_df.at[i, "SEXUAL_HARASSMENT"] = 1 if scores.get("SEXUAL_HARASSMENT", 0) >= threshold else 0
    sample_df.at[i, "HATE_SPEECH"] = 1 if scores.get("HATE_SPEECH", 0) >= threshold else 0
    sample_df.at[i, "OTHER_TOXICITY"] = 1 if scores.get("OTHER_TOXICITY", 0) >= threshold else 0

print(sample_df[[
    "comment",
    "THREAT",
    "BULLYING",
    "SEXUAL_HARASSMENT",
    "HATE_SPEECH",
    "OTHER_TOXICITY"
]].head(10))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

                                             comment  THREAT  BULLYING  \
0  you can almost consider this game abandonware ...       0         0   
1  in my life i have never played a more broken g...       0         0   
2  in my experience, if someone asked me why i he...       0         0   
3  most nights, if someone asked me why i hesitat...       0         0   
4  Every time I queue Street Fighter 6, the playe...       0         0   
5  At this point, New World gets ugly fast. The m...       0         0   
6  my biggest issue is, Valorant has one of the w...       0         0   
7  The matches in DayZ can be great, then the com...       0         0   
8  The rough part of Paladins isn't the grind, it...       0         0   
9  [Community rant] What pushed me away from Call...       0         0   

   SEXUAL_HARASSMENT  HATE_SPEECH  OTHER_TOXICITY  
0                  0            0               0  
1                  0            0               0  
2                  0         

**Block 27**
Similar to Block 26 but takes only the first 30 rows of the validation set, runs prediction, and saves the resulting table as an Excel file predicted_labels_table.xlsx.

In [ ]:
import pandas as pd
from transformers import pipeline

# 1) Save the final model once
trainer.save_model("./final_multilabel_model")
tokenizer.save_pretrained("./final_multilabel_model")

# 2) Load the model
clf = pipeline(
    "text-classification",
    model="./final_multilabel_model",
    tokenizer="./final_multilabel_model",
    top_k=None,
    truncation=True
)

# 3) Select the data you want to inspect
sample_df = val_df.copy().reset_index(drop=True)

# If you only want to display the first 30 rows:
sample_df = sample_df.head(30).copy()

texts = sample_df["comment"].tolist()
preds = clf(texts, batch_size=32)

threshold = 0.4

# 4) Prepare prediction columns
for col in ["THREAT", "BULLYING", "SEXUAL_HARASSMENT", "HATE_SPEECH", "OTHER_TOXICITY"]:
    sample_df[col] = 0

# 5) Convert predictions to 0/1
for i, pred_list in enumerate(preds):
    scores = {x["label"].upper(): x["score"] for x in pred_list}

    sample_df.at[i, "THREAT"] = 1 if scores.get("THREAT", 0) >= threshold else 0
    sample_df.at[i, "BULLYING"] = 1 if scores.get("BULLYING", 0) >= threshold else 0
    sample_df.at[i, "SEXUAL_HARASSMENT"] = 1 if scores.get("SEXUAL_HARASSMENT", 0) >= threshold else 0
    sample_df.at[i, "HATE_SPEECH"] = 1 if scores.get("HATE_SPEECH", 0) >= threshold else 0
    sample_df.at[i, "OTHER_TOXICITY"] = 1 if scores.get("OTHER_TOXICITY", 0) >= threshold else 0

# 6) Arrange the final table
table = sample_df[[
    "comment",
    "THREAT",
    "BULLYING",
    "SEXUAL_HARASSMENT",
    "HATE_SPEECH",
    "OTHER_TOXICITY"
]].copy()

# 7) Clean display
pd.set_option("display.max_colwidth", 120)
display(table)

# 8) For saving
table.to_excel("predicted_labels_table.xlsx", index=False)
print(" Saved file: predicted_labels_table.xlsx")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

,comment,THREAT,BULLYING,SEXUAL_HARASSMENT,HATE_SPEECH,OTHER_TOXICITY
0,you can almost consider this game abandonware valve doesn t give a single shit and has left this game in the hands o...,0,0,0,0,0
1,in my life i have never played a more broken game enemies just dont take damage you get killed behind a wall randoml...,0,0,0,0,0
2,"in my experience, if someone asked me why i hesitate to recommend Sea of Thieves, i would point to the behavior of t...",0,0,0,0,0
3,"most nights, if someone asked me why i hesitate to recommend League of Legends, i would point to the behavior of the...",0,0,0,0,0
4,"Every time I queue Street Fighter 6, the player culture wears me out faster than the matches. The loudest players in...",0,0,0,0,0
5,"At this point, New World gets ugly fast. The moment people from certain countries talk, they watch the chat drift in...",0,0,0,0,0
6,"my biggest issue is, Valorant has one of the worst communities i have seen because arguments escalate into threats f...",0,0,0,0,0
7,"The matches in DayZ can be great, then the community reminds you why you muted comms. It doesn't take much: the mome...",0,0,0,0,0
8,"The rough part of Paladins isn't the grind, it's the people around it. Match chat is mostly yelling, swearing, and b...",0,0,0,0,0
9,"[Community rant] What pushed me away from Call of Duty wasn't balance, it was the way players treat each other. Ther...",0,0,0,0,0


✅ Saved file: predicted_labels_table.xlsx


**Block 28**
Uses a threshold of 0.5 and maps the model’s raw LABEL_0 … LABEL_4 outputs to the five toxicity categories. Also creates an is_toxic column that is 1 if any of the five labels is 1.

In [ ]:
threshold = 0.5

for i, pred_list in enumerate(preds):
    score_map = {item["label"]: item["score"] for item in pred_list}

    sample_df.at[i, "THREAT"] = 1 if score_map.get("LABEL_0", 0) >= threshold else 0
    sample_df.at[i, "BULLYING"] = 1 if score_map.get("LABEL_1", 0) >= threshold else 0
    sample_df.at[i, "SEXUAL_HARASSMENT"] = 1 if score_map.get("LABEL_2", 0) >= threshold else 0
    sample_df.at[i, "HATE_SPEECH"] = 1 if score_map.get("LABEL_3", 0) >= threshold else 0
    sample_df.at[i, "OTHER_TOXICITY"] = 1 if score_map.get("LABEL_4", 0) >= threshold else 0

sample_df["is_toxic"] = (
    sample_df[["THREAT", "BULLYING", "SEXUAL_HARASSMENT", "HATE_SPEECH", "OTHER_TOXICITY"]]
    .sum(axis=1) > 0
).astype(int)

# Final Output Formatting
Selects the final columns (comment, is_toxic, and the five toxicity flags), displays the first 500 rows in a readable format, and saves them to final_output_500.xlsx.

In [ ]:
# Choose only the columns you want
final_table = sample_df[[
    "comment",
    "is_toxic",
    "THREAT",
    "BULLYING",
    "SEXUAL_HARASSMENT",
    "HATE_SPEECH",
    "OTHER_TOXICITY"
]].copy()

pd.set_option("display.max_colwidth", 120)

display(final_table.head(500))

final_table.head(500).to_excel("final_output_500.xlsx", index=False)

print("Saved file: final_output_500.xlsx")

,comment,is_toxic,THREAT,BULLYING,SEXUAL_HARASSMENT,HATE_SPEECH,OTHER_TOXICITY
0,you can almost consider this game abandonware valve doesn t give a single shit and has left this game in the hands o...,0,0,0,0,0,0
1,in my life i have never played a more broken game enemies just dont take damage you get killed behind a wall randoml...,0,0,0,0,0,0
2,"in my experience, if someone asked me why i hesitate to recommend Sea of Thieves, i would point to the behavior of t...",1,0,1,1,1,1
3,"most nights, if someone asked me why i hesitate to recommend League of Legends, i would point to the behavior of the...",1,1,1,0,1,1
4,"Every time I queue Street Fighter 6, the player culture wears me out faster than the matches. The loudest players in...",1,1,1,0,0,1
...,...,...,...,...,...,...,...
495,Naraka: Bladepoint gets ugly socially pretty fast. The argument doesn't stay in-game for long; players throw around ...,1,1,0,0,1,0
496,"in my experience, i can handle losing, but with Valorant, the mechanics are not even my biggest complaint anymore. R...",1,0,1,0,1,0
497,Marvel Rivals would be way easier to love if the social side of it wasn't this consistently ugly. Some matches feel ...,1,0,0,1,0,0
498,Destiny 2 would be a lot more fun if the community wasn't this exhausting. The social feed around the game is mostly...,1,0,0,0,0,1


✅ Saved file: final_output_500.xlsx


**Block 28 (Large training block)**
This block loads the final labeled dataset (final_data_for_processingمعتمد.xlsx), splits it into 80% training and 20% unseen test data, and trains a RoBERTa multi‑label classifier for 5 toxicity categories (threat, bullying, sexual harassment, hate speech, other toxicity). After training, it evaluates the model on the unseen test split, saves the model, and generates predictions for the test set. The predictions are saved as final_unseen_predictions.xlsx which includes the original comment, the binary is_toxic flag, and the five toxicity labels.

In [ ]:
# ============================================================
# FINAL TRAIN + TEST ON UNSEEN SPLIT
# Train on one part, predict labels on the other part
# ============================================================

!pip install -q pandas openpyxl scikit-learn transformers datasets torch accelerate

import os
import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# ============================================================
# STEP 1: Load final dataset
# ============================================================
file_path = "sample_data/final_data_for_processingمعتمد.xlsx"   # غيريه إذا احتجتي path مختلف
df = pd.read_excel(file_path)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

# ============================================================
# STEP 2: Keep only required columns
# ============================================================
df = df[[
    "comment",
    "threat",
    "bullying",
    "sexual_harassment",
    "hate_speech",
    "other_toxicity"
]].copy()

df["comment"] = df["comment"].astype(str).fillna("").str.strip()
df = df[df["comment"].str.len() > 3].copy().reset_index(drop=True)

label_cols = ["threat", "bullying", "sexual_harassment", "hate_speech", "other_toxicity"]
for col in label_cols:
    df[col] = df[col].astype(float)

# ============================================================
# STEP 3: Split into train and unseen test
# ============================================================
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

# ============================================================
# STEP 4: Tokenization
# ============================================================
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["comment"],
        truncation=True,
        padding=True,
        max_length=256
    )

def convert_to_dataset(data):
    ds = Dataset.from_pandas(data.reset_index(drop=True))
    ds = ds.map(tokenize, batched=True)

    def format_labels(example):
        example["labels"] = [
            float(example["threat"]),
            float(example["bullying"]),
            float(example["sexual_harassment"]),
            float(example["hate_speech"]),
            float(example["other_toxicity"])
        ]
        return example

    ds = ds.map(format_labels)

    keep_cols = ["input_ids", "attention_mask", "labels"]
    ds = ds.remove_columns([c for c in ds.column_names if c not in keep_cols])
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

train_ds = convert_to_dataset(train_df)
test_ds = convert_to_dataset(test_df)

# ============================================================
# STEP 5: Load model
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5,
    problem_type="multi_label_classification"
)

# ============================================================
# STEP 6: Metrics
# ============================================================
THRESHOLD = 0.4

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= THRESHOLD).astype(int)

    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    subset_acc = accuracy_score(labels, preds)

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "subset_accuracy": subset_acc
    }

# ============================================================
# STEP 7: Training arguments
# ============================================================
training_args = TrainingArguments(
    output_dir="./final_roberta_multilabel_model",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    report_to="none",
    fp16=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

# ============================================================
# STEP 8: Train
# ============================================================
trainer.train()

# ============================================================
# STEP 9: Evaluate on unseen test split
# ============================================================
results = trainer.evaluate()

print("\n================ FINAL TEST RESULTS ================")
print(results)

# ============================================================
# STEP 10: Predict on unseen test split
# ============================================================
raw_preds = trainer.predict(test_ds)
logits = raw_preds.predictions
probs = 1 / (1 + np.exp(-logits))
preds = (probs >= THRESHOLD).astype(int)

# ============================================================
# STEP 11: Build final prediction table
# ============================================================
final_table = test_df.reset_index(drop=True).copy()

final_table["THREAT"] = preds[:, 0]
final_table["BULLYING"] = preds[:, 1]
final_table["SEXUAL_HARASSMENT"] = preds[:, 2]
final_table["HATE_SPEECH"] = preds[:, 3]
final_table["OTHER_TOXICITY"] = preds[:, 4]

final_table["is_toxic"] = (
    final_table[["THREAT", "BULLYING", "SEXUAL_HARASSMENT", "HATE_SPEECH", "OTHER_TOXICITY"]]
    .sum(axis=1) > 0
).astype(int)

# Final columns
final_table = final_table[[
    "comment",
    "is_toxic",
    "THREAT",
    "BULLYING",
    "SEXUAL_HARASSMENT",
    "HATE_SPEECH",
    "OTHER_TOXICITY"
]]

# Save the result
final_table.to_excel("final_unseen_predictions.xlsx", index=False)

print("\nSaved final predictions to: final_unseen_predictions.xlsx")
print("Final rows:", len(final_table))
display(final_table.head(20))

Rows: 11222
Columns: ['comment', 'is_toxic', 'threat', 'bullying', 'sexual_harassment', 'hate_speech', 'other_toxicity']
Train size: 8975
Test size: 2244


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


================ FINAL TEST RESULTS ================
{'eval_loss': 0.062409330159425735, 'eval_micro_f1': 0.9561752988047809, 'eval_macro_f1': 0.9566223209513904, 'eval_subset_accuracy': 0.9327094474153298, 'eval_runtime': 8.1421, 'eval_samples_per_second': 275.604, 'eval_steps_per_second': 34.512, 'epoch': 5.0}

✅ Saved final predictions to: final_unseen_predictions.xlsx
✅ Final rows: 2244


,comment,is_toxic,THREAT,BULLYING,SEXUAL_HARASSMENT,HATE_SPEECH,OTHER_TOXICITY
0,"Massively underrated, massively difficult, mas...",0,0,0,0,0,0
1,"Don't get me wrong, this game is incredibly bu...",0,0,0,0,0,0
2,"At this point, VRChat gets ugly fast. Match ch...",1,0,0,0,0,1
3,"More often than not, Overwatch ranked gets ugl...",1,0,0,0,1,0
4,Guild Wars 2 gets ugly socially pretty fast. I...,1,1,0,0,0,1
5,the single player is a good exspirence. but i ...,0,0,0,0,0,0
6,WOW!! These girls are killer!! and i mean like...,0,0,0,0,0,0
7,Naraka: Bladepoint gets ugly socially pretty f...,1,0,1,0,1,0
8,EA FC gets ugly socially pretty fast. As soon ...,1,0,0,1,0,1
9,Valorant comp gets ugly socially pretty fast. ...,1,1,0,0,1,0


**Block 29 (Small display block)**
Loads the previously saved predictions file final_unseen_predictions.xlsx, displays the first 500 rows in a formatted table, and saves those 500 rows to a new Excel file named final_output_500.xlsx. This is useful for a quick inspection or for sharing a sample of the results.

In [ ]:
import pandas as pd

# # Load the final file
final_df = pd.read_excel("final_unseen_predictions.xlsx")

# # Display the first 500 rows neatly
pd.set_option("display.max_colwidth", 120)

display(final_df.head(500))

# If you want to save the first 500 separately:
final_df.head(500).to_excel("final_output_500.xlsx", index=False)

print("Saved file: final_output_500.xlsx")
print("Rows shown:", len(final_df.head(500)))

,comment,is_toxic,THREAT,BULLYING,SEXUAL_HARASSMENT,HATE_SPEECH,OTHER_TOXICITY
0,"Massively underrated, massively difficult, massively fun. Just seems to be stuck in the shadows of its technical spi...",0,0,0,0,0,0
1,"Don't get me wrong, this game is incredibly buggy, and unfinished. But the career mode in it is really good and that...",0,0,0,0,0,0
2,"At this point, VRChat gets ugly fast. Match chat makes every queue feel grimy. It kills the mood fast.",1,0,0,0,0,1
3,"More often than not, Overwatch ranked gets ugly fast. There are still way too many games where people speaking anoth...",1,0,0,0,1,0
4,Guild Wars 2 gets ugly socially pretty fast. It gets dark fast because it jumps straight to 'I'll deal with you' ene...,1,1,0,0,0,1
...,...,...,...,...,...,...,...
495,"on the weekend, the worst part of Overwatch 2 is that the player base feels toxic even when nobody is saying anythin...",1,0,1,0,0,1
496,"Not much to review, as there's not a ton to do in the game at this very moment. I've put more money into early acces...",0,0,0,0,0,0
497,first amazing gun feel the kickback in addition to great sounds makes it feel powerful something else than the peash...,0,0,0,0,0,0
498,"This started life as a spin-off iOS game. It's not bad, actually. Average, but not bad. Pick it up cheap.",0,0,0,0,0,0


✅ Saved file: final_output_500.xlsx
✅ Rows shown: 500


**Extra Step: Filter and Show Only Toxic Comments (with statistics)**

This block filters the final predictions to keep only comments classified as toxic (is_toxic = 1). It prints the number of toxic comments in the test set, the total test samples, and the toxic percentage. Then it displays the first 100 toxic comments in a clean, readable table and saves all toxic comments to a new Excel file named final_toxic_only.xlsx. This is useful for analyzing or reviewing only the toxic examples and understanding the toxicity prevalence in the test set.

In [ ]:
# ============================================================
# EXTRA STEP: Show only toxic comments (is_toxic = 1)
# ============================================================

# # Filter only toxic samples
toxic_only = final_table[final_table["is_toxic"] == 1].copy()

print("Toxic comments in test set:", len(toxic_only))
print("Total test samples:", len(final_table))
print("Toxic percentage: {:.2f}%".format(100 * len(toxic_only) / len(final_table)))

# Display the first 100 rows neatly
import pandas as pd
pd.set_option("display.max_colwidth", 120)

display(toxic_only.head(100))

# Save the file
toxic_only.to_excel("final_toxic_only.xlsx", index=False)

print("Saved file: final_toxic_only.xlsx")

🔥 Toxic comments in test set: 1174
📊 Total test samples: 2244
📈 Toxic percentage: 52.32%


,comment,is_toxic,THREAT,BULLYING,SEXUAL_HARASSMENT,HATE_SPEECH,OTHER_TOXICITY
2,"At this point, VRChat gets ugly fast. Match chat makes every queue feel grimy. It kills the mood fast.",1,0,0,0,0,1
3,"More often than not, Overwatch ranked gets ugly fast. There are still way too many games where people speaking anoth...",1,0,0,0,1,0
4,Guild Wars 2 gets ugly socially pretty fast. It gets dark fast because it jumps straight to 'I'll deal with you' ene...,1,1,0,0,0,1
7,"Naraka: Bladepoint gets ugly socially pretty fast. In LFG groups, quieter players in the lobby get mocked until they...",1,0,1,0,1,0
8,"EA FC gets ugly socially pretty fast. As soon as the lobby hears a female voice, people stop playing and start creep...",1,0,0,1,0,1
...,...,...,...,...,...,...,...
168,Rocket League would be way easier to love if the social side of it wasn't this consistently ugly. There are constant...,1,0,1,1,0,0
173,"Whenever I come back to ARK, I remember the community problem almost immediately. I expect tilt; I don't expect the ...",1,0,1,0,0,0
174,Destiny Trials gets ugly socially pretty fast. A lot of matches feel like players who miss a play become the room's ...,1,0,1,0,1,0
176,The ongoing problem with Paladins is the behavior it normalizes. The loudest players in the room usually set the ton...,1,0,1,0,0,1


✅ Saved file: final_toxic_only.xlsx
